In [1]:
import os
import time
import requests
import json
import pandas as pd
from azure.storage.blob import BlobServiceClient
import uuid
from decimal import Decimal
from dotenv import load_dotenv

BASE_URL = "https://api.mfapi.in/mf"  # Base URL to get mutual Fund information about Schema
KUVERA_BASE_URL = "https://mf.captnemo.in/kuvera" # Base URL to get mutual Fund Information

In [192]:
uuid.uuid4()

UUID('efa2c8aa-a253-4e9b-92af-4f4d1f0ea7bc')

In [2]:
load_dotenv()

True

In [ ]:
os.environ

In [4]:
def hit_api_mf(**kwargs):

    '''
        __
        /__)  _  _     _   _ _/   _
        / (   (- (/ (/ (- _)  /  _)
                /
    
        Simple Single Request to API (https://api.mfapi.in/mf)
    '''

    if kwargs.get("scheme_code"):
        if kwargs.get("latest"):
            url = f"{BASE_URL}/{kwargs['scheme_code']}/latest"
        else:
            url = f"{BASE_URL}/{kwargs['scheme_code']}"
    else:
        url = BASE_URL

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Failed to fetch from {url}: {e}")

    try:
        return response.json()
    
    except ValueError:
        raise RuntimeError(f"Invalid JSON received from {url}")

In [6]:
def get_all_scheme_codes():
    '''
        Request all scheme codes for Mutual Funds.
    '''
    return [
        code.get("schemeCode")
        for code in hit_api_mf()
        if code.get("schemeCode")
    ]


In [7]:
get_all_scheme_codes()

[100027,
 100028,
 100029,
 100030,
 100031,
 100032,
 100033,
 100034,
 100035,
 100036,
 100037,
 100038,
 100041,
 100042,
 100043,
 100044,
 100046,
 100047,
 100048,
 100049,
 100051,
 100052,
 100053,
 100054,
 100055,
 100056,
 100057,
 100058,
 100059,
 100060,
 100061,
 100062,
 100063,
 100064,
 100065,
 100066,
 100067,
 100068,
 100069,
 100077,
 100078,
 100079,
 100080,
 100081,
 100082,
 100084,
 100085,
 100086,
 100087,
 100088,
 100089,
 100090,
 100119,
 100120,
 100121,
 100122,
 100123,
 100124,
 100136,
 100150,
 100151,
 100152,
 100153,
 100154,
 100155,
 100156,
 100171,
 100172,
 100173,
 100174,
 100175,
 100176,
 100177,
 100178,
 100179,
 100180,
 100181,
 100182,
 100183,
 100184,
 100185,
 100186,
 100187,
 100188,
 100189,
 100190,
 100191,
 100192,
 100194,
 100195,
 100196,
 100197,
 100198,
 100199,
 100200,
 100201,
 100202,
 100203,
 100218,
 100219,
 100220,
 100221,
 100222,
 100223,
 100233,
 100234,
 100237,
 100238,
 100241,
 100243,
 100244,
 

In [8]:
from typing import List, Dict

In [219]:
def open_file_get_contents(
        run_time_config_file : str
    ):
    '''
        Open and get contents of the specified run-time configuration
    '''
    with open(run_time_config_file, 'r', encoding='utf-8') as f:
        return json.load(f)


In [220]:
open_file_get_contents(r'D:\Mutual_funds\dags\configs\run_time_config.json')

{'100038': '29-08-2025',
 '100037': '29-08-2025',
 '100034': '29-08-2025',
 '100033': '29-08-2025'}

In [ ]:
def list_difference(
    list1, 
    list2
):
    '''
        A helper to Compare to List
    '''
    return [item for item in list1 if item not in list2]

def check_for_new_scheme( config_path: str):
    '''
        Call Api and check runtime config for new Scheme Codes Availabe.
        hits endpoint : "https://api.mfapi.in/mf" to get all metadata
    '''

    all_scheme_codes: List = get_all_scheme_codes()
    current_codes : List = [
        int(item) 
        for item in list(
            open_file_get_contents(config_path).keys()
        )
    ]
    if diff := list_difference(all_scheme_codes,current_codes):
        return diff
    else:
        return []
        

In [11]:
from functools import singledispatch
from datetime import datetime, date

@singledispatch
def _serialize(arg):
	"""
		Fallback if an unsupported type is passed in.
	"""
	raise TypeError(f"Cannot serialize type {type(arg).__name__!r}")

@_serialize.register
def _(arg: date) -> str:
	"""
		If the input is a date, return 'DD-MM-YYYY'.
	"""
	return arg.strftime("%d-%m-%Y")

@_serialize.register
def _(arg: str) -> date:
	"""
		If the input is a string 'DD-MM-YYYY', parse it back into a date.
	"""
	return datetime.strptime(arg, "%d-%m-%Y").date()

In [12]:
def today() -> date:
    """
        Today's Date
    """
    return date.today()

def day_gap(value ) -> bool :
    """
        Compare the value with Today (date.today())
        Get the gap between Date
        Signifies Past Date.
    """
    if not isinstance(value, date):
        value = _serialize(value)
    
    return (today() - value).days

In [ ]:
def get_data_after(
    json_obj: Dict,
    cutoff_date: str
) -> List[Dict[str, str]]:
    
    '''
        Serialize and get data After a certian Cutoff from API if not latest and do a full comparision for the same.
    '''
    
    cutoff = _serialize(cutoff_date) 
    filtered =  [
        entry
        for entry in json_obj.get('data', [])
        if _serialize(entry['date']) > cutoff 
    ]
    new_json : Dict = dict(json_obj)
    new_json["data"] = filtered 

    return new_json

In [15]:
def chunk_list_with_uuid(lst: List, chunk_size: int) -> Dict[str, List]:
    """
        Split `lst` into chunks of at most `chunk_size` elements,
        assign each chunk a random UUID, and return a dict mapping
        uuid (as string) -> chunk list.
    """
    if chunk_size < 1:
        raise ValueError("chunk_size must be >= 1")

    result: Dict[str, List] = {}
    for i in range(0, len(lst), chunk_size):
        chunk = lst[i : i + chunk_size]
        key = str(uuid.uuid4())
        result[key] = chunk

    return result

In [16]:
def update_file_contents(
    updates: Dict,
    config_path,
    *,
    indent: int = 2
) -> None:
    """
        Open and update contents of the specified run-time configuration.
        If writing the JSON fails, dump the `updates` dict to a temp/.txt file.
    """
    data = open_file_get_contents(config_path)
    data.update(updates)
    try:
        with open(config_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=indent)
            f.write('\n')

    except Exception as exc:
        base_dir = os.path.dirname(config_path) or '.'
        temp_dir = os.path.join(base_dir, 'temp')
        os.makedirs(temp_dir, exist_ok=True)

        timestamp = datetime.now().strftime('%Y%m%dT%H%M%SZ')
        dump_path = os.path.join(temp_dir, f'updates_{timestamp}.txt')

        with open(dump_path, 'w', encoding='utf-8') as tf:
            tf.write(json.dumps(updates, indent=indent))
            tf.write('\n')
        raise

In [ ]:
def extract_daily(
    config_path : str,
    threshold : int = 30,
):
    '''
        The Daily Driver.
        if 'search_for_new_schemes' is True then look for new Mutual Funds and Load.
        else Start Daily Extracts.
    '''
    scheme_data = open_file_get_contents(config_path)
    
    required_schemes = { scheme_code : day_gap(latest_update_date)
            for scheme_code, latest_update_date in scheme_data.items() 
            if day_gap(latest_update_date) <= threshold
    }

    tasks = [
        (
            BASE_URL, 
            code, 
            latest_flags
        )
        for code, latest_flags in 
            {
                intermediate : day == 1
                for intermediate, day in required_schemes.items() 
            }.items()
    ] 

    results = {}

    for base_url, scheme_code, latest in tasks:
        
        print(base_url, scheme_code, latest)
        if scheme_code:
            if latest:
                url = f"{base_url}/{scheme_code}/latest"
            else:
                url = f"{base_url}/{scheme_code}"
        else:
            url = base_url

        try:
            response = requests.get(url, timeout=10)
            time.sleep(0.2)
            response.raise_for_status()
        except Exception as e:
            print(f"Problem with {scheme_code}, Error {e}")
            continue

        try:
            results[scheme_code] = response.json()
            print(f'Got response for data Extraction. {scheme_code}--{ response.status_code}')
        except ValueError:
            return scheme_code, RuntimeError(f"Invalid JSON received from {url}")


    ready_to_submit = []
    exclude_updation = []
    include_updation = []

    for key,value in results.items():
        _data = get_data_after(value,scheme_data[key])
        if _data_key := _data.get('data'):
            ready_to_submit.append(
                get_data_after(value,scheme_data[key])
            )
            include_updation.append(key)
        else:
            exclude_updation.append(key)
            print(f'Empty Data for : {key}')


    if ready_to_submit:
        results = chunk_list_with_uuid(ready_to_submit, 1000)

        for uuids, items in results.items():
            try:
                mappings = []
                code_mappings = {}
                for payload in items:
                    fund_scheme_data = payload.get('meta')
                    scheme_code = fund_scheme_data.get("scheme_code")
                    fund_nav_historical =  payload.get('data')
                    first_date = fund_nav_historical[0].get('date')
                    code_mappings[scheme_code] = first_date
                    for item in fund_nav_historical:
                        try:
                            date_str = item.get('date')
                            nav_str  = item.get('nav')
                            nav_date = datetime.strptime(date_str, "%d-%m-%Y").date()
                            nav_val  = Decimal(nav_str)
                            mappings.append({
                                'insert_date' : datetime.now(),
                                'scheme_code': scheme_code,
                                'date': nav_date,
                                'nav': nav_val
                            })
                        except Exception:
                            continue

                try:

                    import io
                    import pandas as pd
                    import pyarrow as pa
                    import pyarrow.parquet as pq
                    from azure.storage.blob import BlobServiceClient

                    dataframe = pd.DataFrame(mappings, columns=['insert_date','scheme_code','date','nav'])

                    dataframe['insert_date'] = pd.to_datetime(dataframe['insert_date']).dt.tz_localize(None)
                    dataframe['date']        = pd.to_datetime(dataframe['date'])  
                    dataframe['scheme_code'] = pd.to_numeric(dataframe['scheme_code'], downcast='integer')
                    dataframe['nav']         = pd.to_numeric(dataframe['nav'], errors='coerce') 

                    if isinstance(dataframe.index, pd.PeriodIndex):
                        dataframe.index = dataframe.index.to_timestamp()

                    table = pa.Table.from_pandas(dataframe, preserve_index=False)  
                    buf = io.BytesIO()
                    pq.write_table(
                        table,
                        buf,
                        compression="zstd", 
                        use_deprecated_int96_timestamps=False
                    )
                    buf.seek(0)

                    account_url   = os.getenv('ACCOUNT_URL')
                    sas_token     = os.getenv('SAS_TOKEN')
                    container_name= os.getenv('CONTAINER_NAME')

                    today = datetime.today()
                    path = (
                        f"daily_extracts/{today.year}/{today.month:02}/{today.day:02}/"
                        f"mf_daily_navs_{uuid.uuid4()}_{today.year}_{today.month:02}_{today.day:02}.parquet"
                    )

                    blob_service_client = BlobServiceClient(account_url, credential=sas_token)
                    container_client = blob_service_client.get_container_client(container_name)
                    container_client.upload_blob(name=path, data=buf.getvalue(), overwrite=False, content_settings=None)


                except Exception as db_err:
                    print(f'Error Processing {uuids}, Error {db_err}')
                
            except Exception as db_err:
                print(f'Error Processing {uuids}, Error {db_err}')


            print(f'Task {uuids} Processed {len(mappings)} records')
        

        update_file_contents(
            {
                str(key):value for key,value in code_mappings.items()
            }, 
            config_path=config_path
        )


In [19]:
extract_daily(r'D:\Mutual_funds\dags\configs\run_time_config.json')

https://api.mfapi.in/mf 100038 False
Got response for data Extraction. 100038--200
https://api.mfapi.in/mf 100037 False
Got response for data Extraction. 100037--200
https://api.mfapi.in/mf 100034 False
Got response for data Extraction. 100034--200
https://api.mfapi.in/mf 100033 False
Got response for data Extraction. 100033--200
Error Processing 7acda970-5cd0-4f6f-811c-edfca6e245a6, Error The specified account is disabled.
RequestId:25263a99-301e-00df-1d9d-1ae1ef000000
Time:2025-08-31T17:32:41.0088731Z
ErrorCode:AccountIsDisabled
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>AccountIsDisabled</Code><Message>The specified account is disabled.
RequestId:25263a99-301e-00df-1d9d-1ae1ef000000
Time:2025-08-31T17:32:41.0088731Z</Message></Error>
Task 7acda970-5cd0-4f6f-811c-edfca6e245a6 Processed 52 records


In [ ]:
import requests
BASE_URL = "https://api.mfapi.in/mf"

all_api_metdata = requests.get(
    url=BASE_URL
)



In [22]:
all_api_metdata.status_code

200

In [ ]:
all_scheme_codes = [
    item.get('schemeCode') for item in all_api_metdata.json()
]  

In [25]:
all_scheme_codes

[100027,
 100028,
 100029,
 100030,
 100031,
 100032,
 100033,
 100034,
 100035,
 100036,
 100037,
 100038,
 100041,
 100042,
 100043,
 100044,
 100046,
 100047,
 100048,
 100049,
 100051,
 100052,
 100053,
 100054,
 100055,
 100056,
 100057,
 100058,
 100059,
 100060,
 100061,
 100062,
 100063,
 100064,
 100065,
 100066,
 100067,
 100068,
 100069,
 100077,
 100078,
 100079,
 100080,
 100081,
 100082,
 100084,
 100085,
 100086,
 100087,
 100088,
 100089,
 100090,
 100119,
 100120,
 100121,
 100122,
 100123,
 100124,
 100136,
 100150,
 100151,
 100152,
 100153,
 100154,
 100155,
 100156,
 100171,
 100172,
 100173,
 100174,
 100175,
 100176,
 100177,
 100178,
 100179,
 100180,
 100181,
 100182,
 100183,
 100184,
 100185,
 100186,
 100187,
 100188,
 100189,
 100190,
 100191,
 100192,
 100194,
 100195,
 100196,
 100197,
 100198,
 100199,
 100200,
 100201,
 100202,
 100203,
 100218,
 100219,
 100220,
 100221,
 100222,
 100223,
 100233,
 100234,
 100237,
 100238,
 100241,
 100243,
 100244,
 